# Cas pratique n°1 — Prédire les survivants du Titanic

**Jour 1 — chapitre 05 : La boîte à outils du Data Scientist**

## Mise en situation

Le dataset Titanic recense les passagers du paquebot (âge, sexe, classe, tarif, port
d'embarquement…) ainsi qu'une variable indiquant s'ils ont survécu. Notre objectif :
prédire la survie d'un passager à partir de ses caractéristiques.

### Où récupérer les vraies données

Ce notebook charge les données via `seaborn` (`sns.load_dataset("titanic")`) pour
fonctionner immédiatement, sans compte ni téléchargement. Pour le TP réel avec les
participants, utilisez plutôt les fichiers officiels de la compétition Kaggle
*Titanic - Machine Learning from Disaster* :
https://www.kaggle.com/c/titanic

Si vous travaillez dans un Kaggle Notebook attaché à cette compétition, remplacez le
chargement ci-dessous par :

```python
train = pd.read_csv("/kaggle/input/titanic/train.csv")
```

Les noms de colonnes du fichier Kaggle officiel diffèrent légèrement de ceux de seaborn
(`Survived`/`Pclass`/`Sex`/`Age`/`SibSp`/`Parch`/`Fare`/`Embarked` en majuscules côté
Kaggle). Une cellule de correspondance est fournie plus bas pour retomber sur les noms
utilisés dans les slides.


In [1]:
import pandas as pd
import seaborn as sns

titanic_raw = sns.load_dataset("titanic")

# On renomme les colonnes pour retrouver exactement le vocabulaire des slides
# (Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Survived)
titanic = titanic_raw.rename(
    columns={
        "survived": "Survived",
        "pclass": "Pclass",
        "sex": "Sex",
        "age": "Age",
        "sibsp": "SibSp",
        "parch": "Parch",
        "fare": "Fare",
        "embarked": "Embarked",
    }
)[
    [
        "Pclass",
        "Sex",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "Embarked",
        "Survived",
    ]
]

titanic.head()


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0


## Étape 1 — Identifier la target

**Question de réflexion :** quelle colonne est la variable à prédire ? Est-ce un problème
de classification ou de régression ?


In [2]:
print(titanic["Survived"].value_counts())
print()
print("Type de target : classification binaire (0 = non survivant, 1 = survivant)")


Survived
0    549
1    342
Name: count, dtype: int64

Type de target : classification binaire (0 = non survivant, 1 = survivant)


**Ce qu'on observe** : `Survived` est bien la target, une variable catégorielle à deux
valeurs (0/1). C'est donc un problème de classification binaire.


## Étape 2 — Lister les features disponibles

**Question de réflexion :** parmi les colonnes restantes, lesquelles pourraient être de
bonnes features pour prédire la survie ?


In [3]:
features_candidates = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
titanic[features_candidates].dtypes


Pclass        int64
Sex             str
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked        str
dtype: object

**Ce qu'on observe** : on a un mélange de variables numériques continues (Age, Fare), de
variables numériques discrètes (Pclass, SibSp, Parch) et de variables catégorielles
(Sex, Embarked). Ce mélange justifiera l'encodage vu en journée 2.


## Étape 3 — Formuler des hypothèses métier

**Question de réflexion :** avant même de modéliser, quelles hypothèses feriez-vous sur
les facteurs de survie ? (le sexe, la classe, l'âge influencent-ils la survie ?)


In [4]:
print("Taux de survie global :", round(titanic["Survived"].mean(), 3))
print()
print("Taux de survie par sexe :")
print(titanic.groupby("Sex")["Survived"].mean())
print()
print("Taux de survie par classe :")
print(titanic.groupby("Pclass")["Survived"].mean())


Taux de survie global : 0.384

Taux de survie par sexe :
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Taux de survie par classe :
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


**Ce qu'on observe** : le taux de survie est nettement plus élevé chez les femmes et chez
les passagers de 1ère classe. C'est cohérent avec l'intuition historique (« les femmes et
les enfants d'abord », et un accès privilégié aux canots pour les premières classes) — ces
deux variables seront très probablement des features discriminantes.


## Étape 4 — Charger le dataset et observer sa structure

**Question de réflexion :** combien de lignes et de colonnes contient le dataset ? Y a-t-il
des types de données inattendus ?


In [5]:
titanic.info()
print()
titanic.describe()


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    str    
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Embarked  889 non-null    str    
 7   Survived  891 non-null    int64  
dtypes: float64(2), int64(4), str(2)
memory usage: 60.9 KB



,Pclass,Age,SibSp,Parch,Fare,Survived
count,891.000000,714.000000,891.000000,891.000000,891.000000,891.000000
mean,2.308642,29.699118,0.523008,0.381594,32.204208,0.383838
std,0.836071,14.526497,1.102743,0.806057,49.693429,0.486592
min,1.000000,0.420000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,20.125000,0.000000,0.000000,7.910400,0.000000
50%,3.000000,28.000000,0.000000,0.000000,14.454200,0.000000
75%,3.000000,38.000000,1.000000,0.000000,31.000000,1.000000
max,3.000000,80.000000,8.000000,6.000000,512.329200,1.000000


## Étape 5 — Repérer les valeurs manquantes et les types de variables

**Question de réflexion :** quelles colonnes ont des valeurs manquantes, et dans quelle
proportion ?


In [6]:
missing = titanic.isnull().sum()
missing_pct = (missing / len(titanic) * 100).round(1)
pd.DataFrame({"valeurs_manquantes": missing, "pourcentage": missing_pct}).sort_values(
    "valeurs_manquantes", ascending=False
)


,valeurs_manquantes,pourcentage
Age,177,19.9
Embarked,2,0.2
Pclass,0,0.0
Sex,0,0.0
SibSp,0,0.0
Parch,0,0.0
Fare,0,0.0
Survived,0,0.0


**Ce qu'on observe** : `Age` a une part significative de valeurs manquantes, et
`Embarked` en a très peu (voire aucune selon la version des données). C'est exactement
le point de départ du TP « remplissage des valeurs manquantes » de demain, journée 2 —
nous ne comblons rien aujourd'hui, nous nous contentons de diagnostiquer.

**À retenir pour le débriefing du TP** : à l'issue de cette première manipulation, chaque
participant a identifié les features candidates, la target, et les premières pistes de
nettoyage — c'est la base du travail de la journée 2.
